# HomeCare 영상 실환경 평가 (Raspberry Pi 촬영 영상)

`yamnet/yamnet_realworld_eval.ipynb`를 참고한 영상용 평가 노트북이다.
**데이터 검증 → 현재 코드 추론/캐시 → 지표 → 변경 전후 → 그래프 → 오류 분석 → CSV/JSON/PNG 저장** 순서로 실행한다.

## 목표와 평가 단위
- YOLO는 사람의 위치를 찾는 객체 검출 모델이고, 낙상·침입·방문자·택배 이벤트는 시간/위치 규칙으로 결정한다.
- 영상·이벤트 종류별 이진 평가: 양성=그 사건이 있음, 음성=그 사건이 없음. 다른 종류 영상을 무조건 음성으로 간주하지 않는다.
- 낙상은 **의심 포함 경보**와 **감지 단계만**을 따로 계산한다. 두 값을 합산하지 않는다.
- 기존 `vision.evaluation.replay`와 `evaluate`를 사용한다. 카메라/서버/환경변수 파일에 접근하지 않고 운영 코드나 임계값을 수정하지 않는다.

## 실행 방법
1. 기존 vision 의존성 및 matplotlib가 있는 Python 커널을 선택한다. 이 노트북은 패키지를 설치하거나 모델을 다운로드하지 않는다.
2. 아래 `LABELS_PATH`를 검수한 정답 CSV로 설정하고 위에서 아래로 실행한다. 현재 PC에서는 2026-10-22 재촬영 라벨을 기본으로 찾는다.
3. 첫 실행은 전체 영상 추론으로 시간이 걸린다. 이후 **영상·모델·감지 코드·설정·실행 라이브러리 해시가 같은 경우만** 캐시를 재사용한다.
4. 다른 PC에서는 원본과 I03 분할 영상 경로를 CSV에서 수정한다. 파일명이나 촬영 JSON만으로 정답을 추측하지 않는다.

## 음성 노트북과 다른 점 / 한계
- 사건별 연속 확률 점수가 없으므로 YOLO 사람 신뢰도를 낙상 점수로 쓰거나 ROC-AUC/임계값 탐색 곡선을 만들지 않는다.
- ESC-50에 대응하는 영상 외부 검증셋은 연결되어 있지 않다. 공개 데이터 성능·교차검증·FA/h를 임의로 만들지 않는다.
- 같은 개발 영상으로 규칙을 개선했으므로 결과는 독립 test 성능이 아니다. 새 촬영 세션으로 검증해야 한다.
- 오프라인은 영상 시각으로 동작한다. Pi의 실제 프레임 공백, 녹화/전송/알림 지연과 다르다.
- I03 분할은 사건마다 상태를 초기화하므로 연속 재진입 시험을 대신하지 않는다. 방문자 음성이 없으면 오탐 성능은 계산 불가다.


## 1. 환경 및 평가 경로
표는 표준 라이브러리로 작성하며 Jupyter에서는 HTML로 보인다. matplotlib/NumPy는 그래프에 사용한다.
출력은 Git 제외 경로 `vision/evaluation/output/realworld_eval/`에 실행별로 보관한다. 원본이나 이전 결과를 덮어쓰지 않는다.


In [ ]:
import sys, json, csv, hashlib, html, platform
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'vision/evaluation/replay.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('저장소 루트 또는 vision 폴더에서 실행하세요.')
sys.path.insert(0, str(ROOT))
from vision.evaluation import replay
from vision.evaluation.evaluate import load_labels, score, write_report

VISION_DIR = ROOT / 'vision'
RECENT = VISION_DIR / 'evaluation/output/retest_20261022'
LABELS_PATH = RECENT / 'labels.csv' if (RECENT / 'labels.csv').exists() else VISION_DIR / 'evaluation/data/labels.csv'
MODEL_PATH = VISION_DIR / 'yolov8n.pt'
SPLIT = 'dev'                       # 독립 검증 라벨을 준비하면 test로 변경
FORCE_REPLAY = False                # True: 기존 캐시가 있어도 다시 추론
BASELINE_PREDICTIONS = RECENT / 'predictions.json'  # 비교하지 않으려면 None
OUTPUT_BASE = VISION_DIR / 'evaluation/output/realworld_eval'
RUN_DIR = OUTPUT_BASE / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
FIG_DIR = RUN_DIR / 'figures'
CACHE_DIR = OUTPUT_BASE / 'cache'
for p in (RUN_DIR, FIG_DIR, CACHE_DIR): p.mkdir(parents=True, exist_ok=True)

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for font in ('Malgun Gothic', 'AppleGothic', 'Noto Sans CJK KR', 'NanumGothic'):
    if font in available_fonts:
        plt.rcParams['font.family'] = font
        break
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

def show_table(records, columns=None):
    if not records:
        print('해당 항목 없음'); return
    columns = columns or list(records[0])
    def fmt(v):
        if v is None: return '—'
        if isinstance(v, float): return f'{v:.3f}' if np.isfinite(v) else '—'
        return str(v)
    try:
        from IPython.display import display, HTML
    except ImportError:
        print(' | '.join(columns))
        for r in records: print(' | '.join(fmt(r.get(c)) for c in columns))
    else:
        head=''.join('<th>'+html.escape(c)+'</th>' for c in columns)
        body=''.join('<tr>'+''.join('<td>'+html.escape(fmt(r.get(c)))+'</td>' for c in columns)+'</tr>' for r in records)
        display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))

def save_csv(name, records):
    if not records:
        (RUN_DIR / name).write_text('status\nno_rows\n', encoding='utf-8-sig'); return
    columns=list(dict.fromkeys(k for r in records for k in r))
    with (RUN_DIR / name).open('w', encoding='utf-8-sig', newline='') as f:
        writer=csv.DictWriter(f, fieldnames=columns); writer.writeheader(); writer.writerows(records)

def savefig(fig, name):
    fig.savefig(FIG_DIR / (name+'.png'), dpi=200, bbox_inches='tight')
    fig.savefig(FIG_DIR / (name+'.svg'), bbox_inches='tight')
    plt.show(); plt.close(fig)

print('정답:', LABELS_PATH)
print('출력:', RUN_DIR)


## 2. 정답·영상 품질 점검
필수 라벨, 시간 범위, 모드 정책, dev/test 중복은 기존 검증기로 확인한다. 영상 길이·크기·FPS와 실제 파일 해시도 기록한다.
녹화 명목 FPS가 Pi 분석 FPS는 아니다. 가변 프레임률/녹화 프레임 누락 여부는 컨테이너 메타데이터만으로 확정할 수 없다.
양성은 사건 시작/허용 마감, 음성은 전체 영상 구간이 필요하다. 현재 라벨의 F03은 의도적 착석 음성, V03은 통과도 방문자 양성이라는 사용자 정책이다.


In [ ]:
import cv2
if not LABELS_PATH.is_file():
    raise FileNotFoundError('검수한 라벨 CSV가 필요합니다. evaluation/labels.example.csv와 README.md를 참고하세요.')
if not MODEL_PATH.is_file():
    raise FileNotFoundError('로컬 YOLO 모델이 필요합니다. 자동 다운로드하지 않습니다.')
if SPLIT not in ('dev', 'test'): raise ValueError('SPLIT은 dev 또는 test')
rows=[r for r in load_labels(LABELS_PATH) if r['split']==SPLIT]
if not rows: raise ValueError('선택한 split의 정답이 없습니다.')
fall_rows=[r for r in rows if r['scenario'].startswith('F')]
if (any(r['category'] not in ('fall_suspect','fall_detected') for r in fall_rows)
        or len({r['resolved_file'] for r in fall_rows})!=len(fall_rows)):
    raise ValueError('F 영상은 파일별 공통 낙상 정답 한 행만 사용하세요. 감지/의심 두 행을 중복 집계하지 않습니다.')
unreviewed=[r['clip_id'] for r in rows if not r.get('review_status','').startswith('reviewed')]
if unreviewed:
    raise ValueError('육안 검수 후 review_status=reviewed 또는 reviewed_sampled로 기록하세요: '+str(unreviewed))
video_info={}
for r in rows:
    path=Path(r['resolved_file'])
    if not path.is_file(): raise FileNotFoundError(f'{r["clip_id"]}: {path}')
    key=str(path)
    if key not in video_info:
        cap=cv2.VideoCapture(key)
        try:
            if not cap.isOpened(): raise ValueError('영상을 열 수 없음: '+path.name)
            fps, n=cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_COUNT)
            if not np.isfinite(fps) or fps<=0 or not np.isfinite(n) or n<=0: raise ValueError('유효하지 않은 영상 메타데이터')
            video_info[key]={'file':path.name, 'sha256':replay.digest(path), 'fps':fps,
                             'frames':int(n), 'duration_s':n/fps, 'width':int(cap.get(3)), 'height':int(cap.get(4))}
        finally: cap.release()
    if abs(video_info[key]['duration_s']-r['duration_s'])>max(.1,1/video_info[key]['fps']):
        raise ValueError('라벨/영상 길이 불일치: '+r['clip_id'])
print('평가 행:',len(rows),'고유 영상/분할:',len(video_info),'세션:',len({r['session'] for r in rows}))
show_table(list(video_info.values()), ['file','duration_s','fps','frames','width','height'])
inventory=[]
for group in sorted({(r['scenario'][0],r['category']) for r in rows}):
    selected=[r for r in rows if (r['scenario'][0],r['category'])==group]
    inventory.append({'시나리오':group[0], '이벤트':group[1], '양성':sum(r['label'] for r in selected),
                      '음성':sum(not r['label'] for r in selected), '행':len(selected)})
show_table(inventory)
save_csv('video_inventory.csv',list(video_info.values()))
save_csv('label_inventory.csv',inventory)


## 3. 현재 코드 추론 및 검증된 캐시
영상·모델·소스·설정·모드·경비·라이브러리 버전이 같을 때만 캐시를 사용한다. 정답은 추론에 사용하지 않고 채점에서만 사용한다.
한 영상의 방문자/택배 행은 동일 모드이면 추론을 공유한다. 각 영상/분할은 감지 상태를 초기화한다.
도중 실패한 영상은 빈 예측으로 점수화하지 않고 중단한다. 진단의 처리시간은 최초 추론 당시 PC의 시간이며 Pi FPS 측정값이 아니다.


In [ ]:
settings=replay.pipeline_settings()
source_paths=sorted(list(VISION_DIR.glob('*.py'))+[VISION_DIR/'evaluation/replay.py',VISION_DIR/'evaluation/evaluate.py'])
sources={p.relative_to(VISION_DIR).as_posix():replay.digest(p) for p in source_paths}
runtime={'python':platform.python_version(), 'platform':platform.platform()}
for name in ('numpy','opencv-python','torch','ultralytics'):
    try: runtime[name]=version(name)
    except PackageNotFoundError: runtime[name]='not-installed'
model_hash=replay.digest(MODEL_PATH)
signature={'model':model_hash,'sources':sources,'settings':settings,'runtime':runtime,'adapter_version':1}
predictions,diagnostics,memory_cache={}, {}, {}
detector=None
for index,r in enumerate(rows,1):
    identity={'signature':signature,'video':video_info[r['resolved_file']]['sha256'],
              'mode':r['mode'],'armed':r['armed'],'duration_s':r['duration_s']}
    identity=json.loads(json.dumps(identity,sort_keys=True))  # ROI tuple도 저장된 JSON과 같은 list로 정규화
    key=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    cache_path=CACHE_DIR/(key+'.json')
    if key in memory_cache:
        payload=memory_cache[key]; origin='shared_this_run'
    elif cache_path.exists() and not FORCE_REPLAY:
        payload=json.loads(cache_path.read_text(encoding='utf-8'))
        if payload.get('identity')!=identity: raise ValueError('캐시 식별 정보 불일치')
        origin='validated_cache'
    else:
        if detector is None:
            from vision.yolo_detector import YOLODetector
            detector=YOLODetector(str(MODEL_PATH),confidence=settings['YOLO_CONFIDENCE'])
        events,diag=replay.replay_clip(r,detector,settings)
        payload={'identity':identity,'events':events,'diagnostics':diag}
        temp=cache_path.with_suffix('.tmp')
        temp.write_text(json.dumps(payload,ensure_ascii=False,allow_nan=False),encoding='utf-8')
        temp.replace(cache_path); origin='fresh_replay'
    memory_cache[key]=payload
    predictions[r['clip_id']]=payload['events']
    diagnostics[r['clip_id']]={**payload['diagnostics'],'origin':origin,'cache_key':key}
    print(f'{index}/{len(rows)} {r["clip_id"]}: {origin}')
# 元の評価器で全ID・カテゴリ・時間を検証。失敗/欠落を未検出に変換しない。
score(rows,predictions)
metadata={'stage':'offline_raw_detection','split':SPLIT,'settings':settings,'source_sha256':sources,
          'model_sha256':model_hash,'labels_sha256':replay.digest(LABELS_PATH), 'runtime':runtime,
          'diagnostics':diagnostics,'videos':video_info,'clock':'CFR frame index / FPS, not Pi wall clock'}
(RUN_DIR/'predictions.json').write_text(json.dumps({'metadata':metadata,'predictions':predictions},ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
# 원본 CSV를 그대로 복사하지 않고 절대 경로를 명시한 실행 스냅샷을 저장한다.
save_csv('labels_snapshot.csv',[{k:v for k,v in {**r,'file':r['resolved_file']}.items() if k!='resolved_file'} for r in rows])
show_table([{'항목':k,'값':v} for k,v in settings.items()])


## 4. 지표와 채점 범위
**Precision**=울린 것 중 실제 사건 비율, **Recall**=실제 사건 중 감지 비율, **F1/F2**=정밀도·재현율 조화평균(F2는 재현율에 더 큰 가중), **MCC**=양·음성 불균형을 고려한 상관 지표.
**시간 내 감지율**은 정답 시작~마감 안 이벤트가 있어야 성공이다. 전체 영상 어딘가에서 출력되면 TP여도 시간 기준에는 실패할 수 있다.
분모가 없으면 `—`로 표시한다. Wilson 95% 구간은 참고 범위이며 같은 촬영 세션의 상관성을 보정하지 않는다.
F/I/V/D 본 평가와 D방문자/V택배 보조 평가를 분리한다. 종류별 표본과 중복이 달라 전체 정확도 하나로 합치지 않는다.


In [ ]:
GROUPS=[('낙상 의심 포함','F','fall_suspect',True),('낙상 감지만','F','fall_detected',False),
        ('침입','I','intrusion_suspect',False),('방문자','V','visitor_detected',False),
        ('택배','D','delivery_suspect',False),('D 방문자(보조)','D','visitor_detected',False),
        ('V 택배(보조)','V','delivery_suspect',False)]
def evaluate_groups(raw_predictions):
    result,details=[],[]
    for title,prefix,category,combined in GROUPS:
        selected=[r for r in rows if r['scenario'].startswith(prefix) and
                  (prefix=='F' or r['category']==category)]
        if not selected: continue
        selected=[{**r,'category':category} for r in selected]
        pred={r['clip_id']:[{**e,'category':'fall_suspect' if combined and e['category']=='fall_detected' else e['category']}
                           for e in raw_predictions[r['clip_id']]] for r in selected}
        summary,clips=score(selected,pred)
        s=summary['categories'][category]; tp,fp,fn,tn=(s[k] for k in ('TP','FP','FN','TN'))
        s.update(group=title, n_pos=tp+fn,n_neg=tn+fp,
                 specificity=tn/(tn+fp) if tn+fp else None,
                 f2=5*tp/(5*tp+4*fn+fp) if 5*tp+4*fn+fp else None)
        result.append(s);details.extend({**r,'group':title} for r in clips)
    return result,details

after,clip_details=evaluate_groups(predictions)
if not after: raise ValueError('F/I/V/D 시나리오의 평가 행이 없습니다.')
metric_columns=['group','n_pos','n_neg','TP','FN','FP','TN','precision','recall','timely_recall','f1','f2','mcc','accuracy']
show_table(after,metric_columns)
print('—는 분모 부재. 방문자 음성이 없으면 specificity/MCC는 계산 불가입니다.')


## 5. 변경 전후 비교 (같은 정답·같은 원본일 때만)
기본 Before는 2026-10-22 자세 변화 후 소실 경로 추가 전의 저장된 예측이다. After는 위에서 실행한 현재 코드다.
비교는 정답 파일 해시와 원본 입력 해시를 검증한 경우에만 수행한다. 기준 결과가 없거나 검증 자료가 부족하면 생략하고 이유를 표시한다.
단순 저장 결과 비교이며 임계값 최적화/독립 검증 결과가 아니다. 새 라벨을 평가할 때는 `BASELINE_PREDICTIONS=None`으로 두거나 해당 라벨의 기준 실행 결과를 지정한다.


In [ ]:
before, before_details, baseline_note = [], [], '기준 실행 미지정'
if BASELINE_PREDICTIONS is not None:
    try:
        baseline=json.loads(BASELINE_PREDICTIONS.read_text(encoding='utf-8'))
        if baseline['metadata']['labels_sha256']!=metadata['labels_sha256']:
            raise ValueError('정답 CSV 해시 불일치')
        # 기준 실행이 보관한 원본 입력 해시를 검사한다. 임의로 파일 이름만 맞추지 않는다.
        recorded_videos=baseline['metadata'].get('videos')
        if recorded_videos:
            for path,info in video_info.items():
                if recorded_videos[path]['sha256']!=info['sha256']:
                    raise ValueError('기준 영상/분할 해시 불일치: '+info['file'])
            verified_note='정답 및 모든 영상/분할 해시 일치.'
        else:
            # 2026-10-22 기존 평가 형식: 원본 해시와 무손실 분할 구간을 검증.
            import re
            manifest_path=BASELINE_PREDICTIONS.parent/'inputs.json'
            if not manifest_path.is_file(): raise ValueError('기준 원본 inputs.json 없음')
            original_manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
            original_by_name={Path(v['file']).name:v for v in original_manifest}
            for path in video_info:
                p=Path(path)
                name=p.stem.rsplit('_part',1)[0]+'.mp4' if '_part' in p.stem else p.name
                ref=original_by_name[name]
                if replay.digest(ref['file'])!=ref['sha256']: raise ValueError('기준 원본 변경: '+name)
                if '_part' not in p.stem:
                    if video_info[path]['sha256']!=ref['sha256']: raise ValueError('현재 영상이 기준과 다름: '+name)
                else:
                    row=next(r for r in rows if r['resolved_file']==path)
                    match=re.search(r'프레임 \[(\d+),(\d+)\)',row.get('notes',''))
                    if not match: raise ValueError('분할 원본 프레임 범위 기록 없음: '+p.name)
                    start,end=map(int,match.groups())
                    original,segment=cv2.VideoCapture(ref['file']),cv2.VideoCapture(path)
                    try:
                        if original.get(5)!=segment.get(5) or int(segment.get(7))!=end-start:
                            raise ValueError('분할 FPS/프레임 수 불일치: '+p.name)
                        original.set(cv2.CAP_PROP_POS_FRAMES,start)
                        for _ in range(start,end):
                            ok1,a=original.read();ok2,b=segment.read()
                            if not ok1 or not ok2 or a.shape!=b.shape or not np.array_equal(a,b):
                                raise ValueError('분할 프레임과 원본 불일치: '+p.name)
                    finally: original.release();segment.release()
            verified_note='정답/원본 해시 및 I03 분할의 모든 디코딩 프레임 일치. 과거 분할은 당시 무손실 검증 기록에 의존.'
        selected_pred={r['clip_id']:baseline['predictions'][r['clip_id']] for r in rows}
        score(rows,selected_pred)
        before,before_details=evaluate_groups(selected_pred)
        baseline_note=verified_note
        (RUN_DIR/'baseline_snapshot.json').write_text(json.dumps(baseline,ensure_ascii=False,indent=2),encoding='utf-8')
    except (OSError,ValueError,KeyError) as exc:
        baseline_note='비교 생략: '+str(exc)
print(baseline_note)
comparison=[]
for a in after:
    b=next((r for r in before if r['group']==a['group']),None)
    if b:
        comparison.append({'group':a['group'],'Recall Before':b['recall'],'Recall After':a['recall'],
                           'F1 Before':b['f1'],'F1 After':a['f1'],'FN Before':b['FN'],'FN After':a['FN'],
                           'FP Before':b['FP'],'FP After':a['FP']})
show_table(comparison)


## 6. 보고서용 그래프
PNG 200dpi와 SVG를 저장한다. 표본 수, 혼동행렬, 성능과 신뢰구간, 변경 전후, 시간 내 감지 지연을 표시한다.
지연은 성공적으로 시간 내 감지된 양성만 포함하므로 미탐을 제외한 빠른 평균으로 전체 성능을 주장하면 안 된다.


In [ ]:
primary=[r for r in after if '(보조)' not in r['group']]
names=[r['group'] for r in primary]; x=np.arange(len(primary))
fig,ax=plt.subplots(figsize=(11,4.5))
ax.bar(x-.18,[r['n_pos'] for r in primary],.36,label='양성',color='#3b82f6')
ax.bar(x+.18,[r['n_neg'] for r in primary],.36,label='음성',color='#94a3b8')
ax.set_xticks(x,names);ax.set_ylabel('영상/구간 수');ax.set_title('그림 1. 평가 표본 구성 (낙상 두 항목은 같은 영상)');ax.legend()
fig.tight_layout();savefig(fig,'fig1_sample_balance')

fig,axes=plt.subplots(1,len(primary),figsize=(3.1*len(primary),3.5),squeeze=False)
for ax,r in zip(axes.ravel(),primary):
    matrix=np.array([[r['TN'],r['FP']],[r['FN'],r['TP']]])
    ax.imshow(matrix,cmap='Blues',vmin=0,vmax=max(1,int(matrix.max())))
    for (i,j),v in np.ndenumerate(matrix):ax.text(j,i,str(v),ha='center',va='center',color='white' if v>matrix.max()/2 else 'black')
    ax.set_xticks([0,1],['미감지','감지']);ax.set_yticks([0,1],['음성','양성']);ax.set_title(r['group']);ax.set_xlabel('예측')
    if not r['n_neg']: ax.set_ylabel('음성 표본 없음')
fig.suptitle('그림 2. 종류별 이진 혼동행렬');fig.tight_layout();savefig(fig,'fig2_confusion_matrices')

fig,axes=plt.subplots(1,3,figsize=(15,4.8))
for ax,metric in zip(axes,['precision','recall','f1']):
    vals=[np.nan if r[metric] is None else r[metric] for r in primary]
    ax.bar(x,vals,color='#2563eb')
    if metric=='recall':
        for i,r in enumerate(primary):
            ci=r['recall_ci95']
            if ci: ax.errorbar(i,r['recall'],yerr=[[max(0,r['recall']-ci[0])],[max(0,ci[1]-r['recall'])]],fmt='none',ecolor='black',capsize=4)
    for i,v in enumerate(vals):
        if np.isfinite(v):ax.text(i,v+.035,f'{v:.2f}',ha='center',fontsize=9)
    ax.set_xticks(x,names,rotation=20);ax.set_ylim(0,1.2);ax.set_title(metric)
fig.suptitle('그림 3. 현재 성능 / Recall의 Wilson 95% 참고 구간');fig.tight_layout();savefig(fig,'fig3_metrics_ci')

if before:
    fig,axes=plt.subplots(1,2,figsize=(12,4.8))
    for ax,metric in zip(axes,['recall','f1']):
        lookup={r['group']:r for r in before}
        ax.bar(x-.18,[np.nan if lookup[r['group']][metric] is None else lookup[r['group']][metric] for r in primary],.36,color='#94a3b8',label='Before')
        ax.bar(x+.18,[np.nan if r[metric] is None else r[metric] for r in primary],.36,color='#2563eb',label='현재 코드')
        ax.set_xticks(x,names,rotation=20);ax.set_ylim(0,1.15);ax.set_title(metric);ax.legend()
    fig.suptitle('그림 4. 동일 정답의 변경 전후 (개발 표본 자체평가)');fig.tight_layout();savefig(fig,'fig4_before_after')

fig,ax=plt.subplots(figsize=(11,4.8))
for i,r in enumerate(primary):
    ds=[d['latency_s'] for d in clip_details if d['group']==r['group'] and d['latency_s'] is not None]
    if ds:ax.scatter([i]*len(ds),ds,s=45,alpha=.75)
ax.set_xticks(x,[f"{r['group']}\n시간 내 {r['latency_n']}/{r['n_pos']}" for r in primary])
ax.set_ylabel('사건 시작 → 감지 (영상 초)');ax.set_title('그림 5. 시간 내 감지된 양성의 지연 (Pi/서버 지연 아님)')
fig.tight_layout();savefig(fig,'fig5_latency')


## 7. 세션·시나리오·조건별 분해 및 남은 오류
동일 세션을 쪼개 일반 성능으로 해석하지 않는다. 거리·조명이 unknown이면 조건별 성능을 주장할 수 없다.
아래 오류에는 FN/FP와 시간 마감 실패를 포함한다. 출력 reason은 감지 경로이지 미탐 원인 진단이 아니다.
사람 검출 실패/추적 실패/규칙 미충족을 구분하려면 별도의 프레임별 검출 기록과 육안 검수가 필요하다.


In [ ]:
breakdowns=[]
for field in ('session','scenario','lighting','distance'):
    for group in names:
        selected=[r for r in clip_details if r['group']==group and r['kind']=='clip']
        for value in sorted({r[field] for r in selected}):
            sub=[r for r in selected if r[field]==value]; counts=Counter(r['outcome'] for r in sub)
            npos=sum(r['label'] for r in sub)
            breakdowns.append({'group':group,'조건':field,'값':value,'n':len(sub),
                               'TP':counts['TP'],'FN':counts['FN'],'FP':counts['FP'],'TN':counts['TN'],
                               'Recall':counts['TP']/npos if npos else None})
show_table(breakdowns)
errors=[]
for d in clip_details:
    if d['outcome'] in ('FN','FP') or d['timely_hit'] is False:
        events=predictions[d['clip_id']]
        errors.append({'group':d['group'],'clip_id':d['clip_id'],'결과':d['outcome'],
                       '시간 내':d['timely_hit'],'정답 시작':d['start_s'],'마감':d['end_s'],
                       '실제 출력':'; '.join(f"{e['category']}@{e['time_s']:.2f}s({e.get('reason','')})" for e in events) or '없음',
                       '원인':'확인 필요 (의심만 출력된 영상은 감지만 평가에서 FN일 수 있음)'})
show_table(errors)


## 8. 최종 표·재현 정보 저장
`summary.csv`, `comparison.csv`, `clips.csv`, `errors.csv`, `breakdowns.csv`, `video_inventory.csv`, `labels_snapshot.csv`, `predictions.json`, `summary.json`, `figures/*.png`/`*.svg`를 저장한다.
실행 결과에는 로컬 영상 경로가 들어 있으므로 Git 제외 output에 보관한다. 노트북을 커밋할 때 출력은 지운다.
자동으로 임계값을 고르거나 소스에 반영하지 않는다. 개선 후에는 새 세션에서 고정된 코드로 평가해야 한다.


In [ ]:
save_csv('summary.csv',after)
save_csv('comparison.csv',comparison)
save_csv('clips.csv',clip_details)
save_csv('errors.csv',errors)
save_csv('breakdowns.csv',breakdowns)
(RUN_DIR/'summary.json').write_text(json.dumps({'metadata':metadata,'baseline_note':baseline_note,
    'before':before,'after':after},ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
write_report(RUN_DIR/'raw_categories',rows,predictions,metadata)
# raw_categories는 이벤트명 그대로 채점하므로 fall_suspect와 fall_detected를 합치지 않는다.
# 발표에는 분리/통합 의미를 명시한 summary.csv를 사용한다.
headers=['이벤트','양성/음성','TP/FN/FP/TN','Recall','Precision','F1','시간 내 감지']
table=[]
for r in primary:
    fmt=lambda v:'—' if v is None else f'{v:.1%}'
    table.append([r['group'],f"{r['n_pos']}/{r['n_neg']}",f"{r['TP']}/{r['FN']}/{r['FP']}/{r['TN']}",
                  fmt(r['recall']),fmt(r['precision']),f"{r['f1']:.3f}" if r['f1'] is not None else '—',fmt(r['timely_recall'])])
fig,ax=plt.subplots(figsize=(14,.6*(len(table)+3)));ax.axis('off')
tab=ax.table(cellText=table,colLabels=headers,loc='center',cellLoc='center')
tab.auto_set_font_size(False);tab.set_fontsize(10);tab.scale(1,1.8)
for (i,j),item in tab.get_celld().items():
    if i==0:item.set_facecolor('#dbeafe');item.set_text_props(weight='bold')
ax.set_title('그림 6. 현재 영상 평가 요약 (개발 표본, 서버 알림 성공률 아님)',pad=16)
savefig(fig,'fig6_summary_table')
lines=['# 영상 실환경 평가 결과','',f'평가 split: {SPLIT}',f'평가 행: {len(rows)} / 고유 영상·구간: {len(video_info)}',
       '',baseline_note,'','| '+' | '.join(headers)+' |','|'+'|'.join(['---']*len(headers))+'|']
lines += ['| '+' | '.join(r)+' |' for r in table]
lines += ['','낙상 의심 포함/감지만은 같은 영상의 별도 지표이며 합산하지 않습니다.',
          '작은 개발 표본이고 I03은 분할 상태 초기화입니다. 방문자 음성 부재 시 오탐 성능은 확인할 수 없습니다.',
          'Pi 실제 FPS·서버 전달 지연·장시간 무사건 오경보는 별도 검증이 필요합니다.',
          'YOLO 신뢰도는 이벤트 확률이 아니므로 ROC-AUC/자동 임계값 최적화를 수행하지 않았습니다.']
(RUN_DIR/'report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('저장 완료:',RUN_DIR)


## 9. Pi 실시간 성능과 다음 촬영
- 현재 영상 코드에서 거실 경비 OFF/ON 모두 낙상 분석, 침입은 거실 ON만, 방문자·택배는 현관에서 동작한다.
- Pi는 실제 처리 시각을 사용한다. 긴 처리 공백이 후보를 초기화할 수 있으므로 이 노트북의 5FPS 재생 성공과 실기 성공을 동일시하지 않는다.
- 세션별 새 모의 낙상/정상 동작, 단순 퇴장, 숙인 뒤 가림, 화면 가장자리, 누운 자세 방향을 별도 기록해 검증한다.
- 장시간 사건 없는 영상을 background 정답으로 수집해야 FA/h를 평가할 수 있다. 짧은 음성 클립의 FP 비율로 시간당 오경보를 추정하지 않는다.
- 바꾸기 전에는 Before 결과를 보관하고, 개발 데이터로 수정한 뒤에는 독립 촬영 test 데이터로 최종 확인한다.
